# Custom Model Classes #

Users may want to define their own custom parameterisations and create new retrieval schemes for their specific purposes. Implementing new parameterisations in archNEMESIS is relatively straightforward, at the simplest level you just need to add a file containing the new class to one of the `.../Models/PreRTModels/`, or *`.../Models/PostRTModels/` directories. 

We are in the process of rationalising the model classes, so we recommend using the below model classes as a guide:
* In the `.../Models/PreRTModels/` directory:
  - *Model2* in `model_2.py`
  - *Model3* in `model_3.py`
  - *Model9* in `model_9.py`
  - *Model32* in `model_32.py`
  - *Model45* in `model_45.py`
  - *Model47* in `model_47.py`
  - *Model444* in `model_444.py`
* In the `/Models/PostRTModels/` directory:
  - *Model231* in `model_231.py`

NOTE: In each directory there is a template model class, that is the result of the steps (2) and (3) (detailed below), that has the methods that need to be implemented. The body of each method body consists of `raise NotImplementedError('This is a template model and should never be used')` as the first expression and some example code with explanatory comments, just copy the template model class, rename it, give it a unique ID, and replace the method bodies with the required code for your model.

The templates model classes are:
* *TemplatePostRTModel* in the `.../Models/PreRTModels/_template.py` file
* *TemplatePreRTModel* in the `.../Models/PostRTModels/_template.py` file

In general it is easiest to look at the implemented models and follow their example, however more detailed guidance is given below if required.


## Model Definition Guide ##

All model class must inherit from `ModelBase` at some point, usually either through `PreRTModelBase` or `PostRTModelBase`.

For ease of use, model classes should be [*dataclasses* ](https://docs.python.org/3/library/dataclasses.html). This allows us
to define instance attributes in the class body instead of having to make an `__init__()` method. 

Having instance attributes in the class body means we can easily see exactly what attributes the model has and what they are for,
instance attributes can be one of three types:

* StateParam - parameters that are retrieved by ArchNemesis, these are packed into the state vector.

* ConstParam - parameters that are not retrieved by ArchNemesis, they are stored on the model instance.

* VarParam - parameters that are not retrieved by ArchNemesis, they are stored on the model instance and 
             in the `varparams` array. These are used when porting over old FORTRAN-NEMESIS code as 
             `varparams` is used by FORTRAN-NEMESIS to make various files (e.g. *.mre files and bookmarks).
             However, only floating-point (or easily converted) values can be saved here. Therefore, for new
             models using ConstParam is recommended.


Values are assigned to `StateParam`, `ConstParam`, and `VarParam` attributes when the model class is constructed,
the helper class method `from_arrays(...)` is available to assist. The param classes are defined in `.../Models/param.py`.


See below for more information on `StateParam`, `ConstParam`, and `VarParam`


### StateParam ###

To define an instance attribute as a `StateParam`, set the type-hint as below:

    `StateParam.using(<slice>, <description>, <unit (optional)>)`

where:

    <slice> - A python `slice` object that denotes where in the model's region of the state vector the StateParam's
              value is stored. NOTE: This determines where in the state vector a StateParam's value is stored.
    
    <description> - A string that describes the intended use of the attribute. Will appear in various diagnostic information.
    
    <unit (optional)> - A string representation of the unit, or "UNKNOWN" if not set. Will appear in various diagnostic information.


A `StateParam` object has the following members:

    stateparam.slice - Set when defined via `<slice>`
    
    stateparam.description - Set when defined via `<description>`
    
    stateparam.unit - Set when defined via `<unit>`
    
    stateparam.v - The value held by the `StateParam`, this is what gets packed into the state vector.
                   NOTE: packing into and unpacking from the state vector handles log and 
                   un-log operations, so this always holds the un-logged value.
    
    stateparam.e - The error held by the `StateParam`, this is what gets packed into the covariance matrix.
                   NOTE: packing into and unpacking from the covariance matrix handles log and
                   un-log operations, so this always holds the un-logged value.
    
    stateparam.log - A boolean flag, if `True` will store the logarithm of `stateparam.v` and `stateparam.e`
                     in to the state vector and covariance matrix respecitvely. If `False` stores raw value.
                     Defaults to `True`.
    
    stateparam.num_diff - A boolean flag, if `True` will use numerical differentiation, if `False` will use
                          analytical differentiation (if available). Defaults to `False`



### ConstParam and VarParam ###

To define an instance attribute as a `ConstParam` or `VarParam`, set the type-hint to one of the following:

    `ConstParam[<type>].using(<description>, <units (optional)>)`

    `VarParam[<type>].using(<description>, <units (optional)>)`

where:

    <type> - The type of the variable, NOTE: The type is not checked at any point (python is dynamically typed), so be on guard
             when using `VarParam` as the value may be a `float` unless you have explicity cast it somewhere else.

    <description> - A string that describes the intended use of the attribute. Will appear in various diagnostic information.
    
    <unit (optional)> - A string representation of the unit, or "UNKNOWN" if not set. Will appear in various diagnostic information.


A `ConstParam` object and a `VarParam` object have the same members:

    xparam.description - Set when defined via `<description>`
    
    xparam.unit - Set when defined via `<unit>`
    
    xparam.v - The value stored in the `ConstParam` or `VarParam` object. NOTE: this **should** be of the same
               type as when defined via `<type>`, however no checks are made.

## Model Class Call Graph ##

Each **model** class has its methods called in a predictable fashion by ArchNEMESIS. The below lists how methods of a **model** class are called in the order in which they are called during a typical ArchNEMESIS execution.


```

M - Model class
m - Instance of model class


[ArchNEMESIS starts]
  |
  |- [Read Input Files]                                            # Only run if bookmark is not loaded
  |  |
  |  |- Variables_0::read_par(...)
  |  |  |
  |  |  |- M.from_apr_to_state_vector(...)                         # Class method, provided by ModelBase
  |  |  |  |
  |  |  |  |- M.is_varident_valid(...)                             # Class method, provided by PreRTModelBase or PostRTModelBase
  |  |  |  |
  |  |  |  |- M.from_apr_file(...)                                 # Class method, MUST BE IMPLEMETED BY MODEL CLASS
  |  |  |  |  |
  |  |  |  |  |- [Custom code goes here]
  |  |  |  |
  |  |  |  |- m.get_n_stateparam_entries()                         # Instance method, provided by ModelBase
  |  |  |  |
  |  |  |  |- m.set_state_vector_region(...)                       # Instance method, provided by StateVectorModifier
  |  |  |  |
  |  |  |  |- m.push_to_state_vector(...)                          # Instance method, provided by StateVectorModifier
  |  |  |  |
  |  |  |  |- m.push_to_covariance_matrix(...)                     # Instance method, provided by StateVectorModifier
  |  |  |  |
  |  |  |  |- m.push_to_numerical_differentiation_vector(...)      # Instance method, provided by StateVectorModifier
  |  |  |  |
  |  |  |  |- m.varparam_write(...)                                # Instance method, provided by ModelBase
  |  |  |
  |  |  |- m.info()                                                # Instance method, provided by ModelTreePrinter
  |  |
  |- [Finish reading input files]
  |
  |- [Load bookmark]                                               # Not run if input files are read
  |  |
  |  |- M.from_bookmark(...)                                       # Class method, MUST BE IMPLEMENTED BY MODEL CLASS
  |  |  |
  |  |  |- [Custom code goes here]
  |
  |- [Finish load bookmark]
  |
  |- [Perform Retrieval Loop]
  |  |
  |  |- ForwardModel_0::subprofretg(...)
  |  |  |
  |  |  |- m.calculate_from_subprofretg(...)                       # Instance method, default provided by PreRTModelBase but is often overwritten
  |  |  |  |
  |  |  |  |- [Custom code goes here]
  |  |  |  |
  |  |  |  |- m.pull_from_state_vector(...)                        # Instance method, provided by StateVectorModifier, must be called at some point here
  |  |  |  |
  |  |  |  |- [Custom code goes here]
  |  |  |  |
  |  |  |  |- m.calculate(...)                                     # Instance method, MUST BE IMPLEMENTED BY MODEL CLASS
  |  |  |  |  |
  |  |  |  |  |- [Custom code goes here]
  |  |  |  |
  |  |  |  |- [Custom code goes here]
  |  |  |
  |  |- ForwardModel_0::subspecret(...)
  |  |  |
  |  |  |- m.calculate_from_subspecret(...)                        # Instance method, default provided by PostRTModelBase but is often ovewritten
  |  |  |  |
  |  |  |  |- [Custom code goes here]
  |  |  |  |
  |  |  |  |- m.pull_from_state_vector(...)                        # Instance method, provided by StateVectorModifier, must be called at some point here
  |  |  |  |
  |  |  |  |- [Custom code goes here]
  |  |  |  |
  |  |  |  |- m.calculate(...)                                     # Instance method, MUST BE IMPLEMENTED BY MODEL CLASS
  |  |  |  |  |
  |  |  |  |  |- [Custom code goes here]
  |  |  |  |
  |  |  |  |- [Custom code goes here]
  |
  |- [End Retrieval Loop]
  |
[ArchNEMESIS Ends]


```

## General Steps

1. Work out what kind of model you are creating, this informs where you put your new model class.
  - Models applied before the radiative transfer calculation are in `.../Models/PreRTModels/`, there are generally two types:
    - Atmospheric models affect one of the following atmosphere profiles: 
      * gas volume mixing ratio
      * aerosol species (dust) density
      * temperature
      * para-h2 fraction
      * fractional cloud cover
    - Non-Atmospheric models affect other things, e.g.:
      * Optical properties of an aerosol species
      * Instrument lineshape
      * Doppler shift of the observer w.r.t the target
  - Models applied after the radiative transfer calculation are in `.../Models/PostRTModels/`*, currently there is only one type:
    - Spectral models affect the the modelled output spectrum directly.

2. Look at the *ModelBase* class in the `.../Models/ModelBase.py` file. List any **abstract methods** on the class (they are marked with `@abstractmethod`). At the time of writing the list of **abstract methods** was:
  - `is_varident_valid`
  - `from_apr_file`
  - `calculate`
  - `calculate_from_subprofretg`
  - `calculate_from_subspecret`

3. In the directory that corresponds to the type of model you are creating, look at the "model type base class": *PreRTModelBase* in `.../Models/PreRTModels/_base.py`; *PostRTModelBase* in `'.../Models/PostRTModels/_base.py`. Do the following:
   1) Any methods with the same name as an **abstract method** of the *ModelBase* class does **NOT** have to be implemented, as the "model type base class" already provides an implementation for you. Have a look at the provided implementation, if it does what you want it to do, you can cross it off your list.
   
   2) Any **abstract methods** on the "model type base class" (they are marked with `@abstractmethod`), **DO** have to be implemented. Add these to your list.

4. Write a new class that implements all of the **abstract methods** on your list, and give it an integer class attribute `id` that is a unique ID number that will be used to identify your custom model, there is a test to ensure all models have a unique ID number so run the tests after adding a model to ensure everything is set up properly. Use the other models in the folder as a guide.





## Example Custom Model Class

As it is much easier to see the above process in action, we present a walkthrough example of implementing a custom model. The example we are using is not going to be *useful*, but it serves as a guide that shows how to do something non-trivial.

Our custom model will parameterise the volume mixing ratio of a gas as a stepwise function. This is not really that *useful*, but it gives us the opportunity to explore how to do a few different things.

Following the above procedure:

* Step 1 is to work out what kind of model we are creating. In this case the model is an Atmospheric model (as it alters a gas volume mixing ratio), so we will put the model in the `.../Models/PreRTModels/` directory.

* Step 2 is to look at *ModelBase* and *PreRTModelBase* and write down the **abstract methods** that require an implementation. At the time of writng (30/06/2025), these are the relevant methods on the two classes:
  
  - *ModelBase* **abstract methods**:
    - `is_varident_valid`
    - `from_apr_file`
    - `calculate`
    - `calculate_from_subprofretg`
    - `calculate_from_subspecret`
  
  - *PreRTModelBase* **concrete methods** (i.e. non-abstract):
    + `is_varident_valid`
    + `calculate_from_subprofretg`
    + `calculate_from_subspecret`
  
  - *PreRTModelBase* **abstract methods**:
    + NO ABSTRACT METHODS

Therefore, if we combine the **abstract methods** from both classes, and then remove the **concrete methods** of *PreRTModelBase*, we have the following list of methods that our new class needs to implement: `from_apr_file`, `calculate`. However, the provided implementation of `calculate_from_subprofretg` is not what we want to use so we need to overwrite this method too. This is step 3.

We can use the template class in `.../Models/PreRTModels/_base.py`* to speed up steps (2) and (3), we just copy the template class, rename it, and re-write its methods to do what we need.

Step 4 is to actually implement the methods, and is performed in the following sections.


### first definitions of the model class

First off, we need a name. The current model names are things like *Model45*, we want a nice name so we will use *Model69* and the file will be `.../Models/PreRTModels/model_69.py`.

We need an ID number, we should choose something that is somewhat consistent with the current ID numbers if possible. I will use 69 for this example.

We should add a docstring to the class that describes it, and we should do some imports.

Finally, we need to know what information the model must hold on to. We want a piecewise representation of a gas volume mixing ratio, therefore we need to save the number of pieces, the value of the piecewise parts, and the pressure of the piece edges.

The number of pieces and their edges will not change over the course of a retrieval, so we can use a `ConstParam`. The value of the piecewise parts are what we want to retrieve so they should be `StateParam` attributes.

At this point our model class looks like (some code omitted for clarity):
```python
from typing import ClassVar, Self
import dataclass as dc

from ..param import (
   StateParam,
   ConstParam,
   VarParam
)

@dc.dataclass
class Model69(PreRTModelBase):
   """
   Parameterises the gas volume mixing ratio as a piecewise function with `n_chunk` chunks.
   The chunk values are retrieved.
   """
   id : ClassVar[int] = 69
   
   piecewise_profile : StateParam.using('Value of the piecewise profile', 'RATIO')
   
   n_pieces : ConstParam[int].using('Number of chunks in the piecewise profile')
   pressure_edges : ConstParam[np.ndarray].using('Pressure at the edges of each piecewise part', 'atm')
   
   
   @classmethod
   def from_apr(
      # OMITTED CODE #

   def calculate_from_subprofretg(
      # OMITTED CODE #
   
   def calculate(
      # OMITTED CODE #
```

### writing `from_apr`

Inspecting the `from_apr` method in `BaseModel` shows what arguments are passed to us, and what we should return.

We are passed:

```
   f : IO,                                    # The open file descriptor of the *.apr file
   varident : np.ndarray[[3],int],            # The three "varident" integers that were last read from the *.apr file
   npro : int,                                # Number of altitude levels defined for the Atmosphere component
   ngas : int,                                # Number of gas volume mixing ratio profiles defined for the reference atmosphere
   ndust : int,                               # Number of aerosol species density profiles define for the reference atmosphere
   nlocations : int,                          # Number of locations defined for the atmosphere component
   runname : str,                             # Name of the *.apr file, without extension.
   sxminfac : float,                          # Minimum factor to bother calculating covariance matrix entries, below this value entries will be zero
   input_file_type : ArchNemesisFileTypeEnum, # Type of input files that we are reading the model from.
```

We are expected to return an instance of the model class. Packing values into the **state vector** is handled in `BaseModel::from_apr_to_state_vector(...)`, so we don't need to do it here.

We can decide how the values are packed into the `<runname>.apr` file, let us say the first line should be the number of piece, then a line for the minimum pressure, followed by lines containing the max pressure of a chunk the apriori VMR and apriori error.

Expected input format specified below should be added as a string on the `apr_input_format` attribute.
```
n_pieces
p_min
p_max[0] vmr[0] vmr_err[0]
p_max[1] vmr[1] vmr_err[1]
...
p_max[n_pieces] vmr[n_pieces] vmr_err[n_pieces]

```

We can use some helper functions to make our lives easier, the `from_apr` method is:

```python

class Model69:

   # code omitted
   
   id : ClassVar[int] = 69
   apr_input_format : ClassVar[str] = \
   """
      n_pieces
      p_min
      p_max[0] vmr[0] vmr_err[0]
      p_max[1] vmr[1] vmr_err[1]
      ...
      p_max[n_pieces] vmr[n_pieces] vmr_err[n_pieces]
   
   """
   
   piecewise_profile : StateParam.using('Value of the piecewise profile', 'RATIO')
   
   n_pieces : ConstParam[int].using('Number of chunks in the piecewise profile')
   pressure_edges : ConstParam[np.ndarray].using('Pressure at the edges of each piecewise part', 'atm')

   @classmethod
   def from_apr(
      cls,
      f : IO,                                    # The open file descriptor of the *.apr file
      varident : np.ndarray[[3],int],            # The three "varident" integers that were last read from the *.apr file
      npro : int,                                # Number of altitude levels defined for the Atmosphere component
      ngas : int,                                # Number of gas volume mixing ratio profiles defined for the reference atmosphere
      ndust : int,                               # Number of aerosol species density profiles define for the reference atmosphere
      nlocations : int,                          # Number of locations defined for the atmosphere component
      runname : str,                             # Name of the *.apr file, without extension.
      sxminfac : float,                          # Minimum factor to bother calculating covariance matrix entries, below this value entries will be zero
      input_file_type : ArchNemesisFileTypeEnum, # Type of input files that we are reading the model from.
   ) -> Self:
      
      # Read from *.apr file
      n_peices = cls.read_apr_entries(f, (int,))
      p_edges = np.zeros((n_peices+1,))
      
      p_edges[0] = cls.read_apr_entries(f, (float,))
      p_edges[1:], xvals, xerrs = cls.read_apr_entries(f, (float, float, float), n_peices)
      
      # Construct model instance
      instance = cls.from_arrays(
         xvals,
         xerrs,
         n_peices,
         p_edges,
      )
      
      # Set model state parameter flags
      instance.piecewise_profile.log = True # `True` is the default, but set it here anyway
      instance.piecewise_profile.num_diff = True #  Always use numerical differentiation

      # return the model instance
      return instance
   
   # code omitted

```


### writing `calculate_from_subprofretg`

This method is called from `ForwardModel_0::subprofretg` and is how an atmospheric model influences the state of the retrieval. We have to perform a few tasks in this method.

1) Unpack parameters from the state vector and get other values we will need
2) Use the `self.calculate(...)` method (see later) to calculate the gas volume mixing ratio according to our model
3) Put the results of the calculation in their correct palce.

The `forward_model` argument is the current `ForwardModel_0` instance, we can access all of its components using this argument. It is very useful as in principle a model class may alter practically anything about a retrieval by interacting with this argument.

In our case we only have one parameter, and there is a helpful method `self.pull_from_state_vector` (defined on the `StateVectorModifier` class that we have indirectly inherited from) that we can use to set all of the `StateParam` attribute values to the values in the **state vector**. We need to pass the state vector and the state vector log flags to this method. Those are stored on the `Variables` component of the `forward_model` as `forward_model.Variables.XN` and `forward_model.Variables.LX` respectively.

We can also find the index of the gas volume mixing ratio we need to alter. There are actually two ways to do this. The first uses the `ipar` argument, which encodes the profile type and profile index that an atmospheric profile should alter. We can get the profile type and index from `ipar` using the `Atmosphere_0::ipar_to_atm_profile_type(...)` method on the `forward_model.AtmosphereX` object. 

We can also make sure we have been given the correct profile type. Checks like this may seem superfulous, but they make sure that if something does go wrong the error is caught as early as possible, which makes debugging much easier. The profile types are defined in the `.../archnemesis/enum/atmospheric_profile_type_enum.py` file.

The `calculate_from_subprofretg` method:
```python

from archnemesis.enum import AtmosphericProfileTypeEnum

class Model69:

   # code omitted

   def calculate_from_subprofretg(
      self,
      forward_model : "ForwardModel_0",
      ix : int,
      ipar : int,
      ivar : int,
      xmap : np.ndarray,
    ) -> None:
      atm = forward_model.AtmosphereX
      atm_profile_type, atm_profile_idx = atm.ipar_to_atm_profile_type(ipar)
      
      assert atm_profile_type == AtmosphericProfileTypeEnum.GAS_VOLUME_MIXING_RATIO
      
      self.pull_from_state_vector(forward_model.Variables.XN, forward_model.Variables.LX) # NOTE: this un-logs values if required
      
      atm, xmap1 = self.calculate(
         atm,
         atm_profile_type,
         atm_profile_idx,
      )
      
      forward_model.AtmosphereX = atm
      xmap[self.state_vector_slice, ipar, 0:atm.NP] = xmap1
   
   # code omitted

```


#### write `self.calculate(...)`

We have a lot of freedom to define how the class method `self.calculate` works. However, the conventions used by the other models are a good place to start. We can always alter the method later if we need to. For now we will send the atmosphere class, the profile type and index. We will assume that we will get back an updated atmosphere class and an array of functional derivatives.

The `calculate` method:
```python

from archnemesis.enum import AtmosphericProfileTypeEnum

class Model69:

   # code omitted

   def calculate(
         self, 
         atm : "Atmosphere_0",
         #   Instance of Atmosphere_0 class we are operating upon
         
         atm_profile_type : AtmosphericProfileTypeEnum,
         #   ENUM of atmospheric profile type we are altering.
         
         atm_profile_idx : int | None,
         #   Index of the atmospheric profile we are altering (or None if the profile type does not have multiples)
   ) -> tuple["Atmosphere_0", np.ndarray]:
      p_bins = self.pressure_edges.v
      p_vals = self.piecewise_profile.v
   
      # update profile values
      temp = np.array(atm.VMR)
      fidx = np.digitize(atm.P, p_bins)
      
      temp[fidx] = p_vals[fidx]
      atm.edit_VMR(temp)
      
      # find the functional derivatives
      xmap = np.zeros((chunk.size,atm.VMR.size[0]),float)
      j=0
      for i in range(chunk.size):
         while fidx[j] < i+1:
            xmap[i, j] = 1
            j += 1

      # return results
      return atm, xmap
   
   # code omitted

```


### Full final code listing

Below is the full code listing for this example model class. You can add it to the `../Models/PreRTModels/` directory and run `python -c 'from archnemesis.Models import Models; print(Models.info(69))'` to see a summary of this model. 

NOTE: This is an example model created as an learning aid. It has not been tested so the exact implementation may have some bugs, but the process of creation is correct. Whenever creating a new model, think about adding a test to check the new model performs as expected.

```python
from typing import ClassVar, Self
import dataclass as dc

from ..param import (
   StateParam,
   ConstParam,
   VarParam
)

@dc.dataclass
class Model69(PreRTModelBase):
   """
   Parameterises the gas volume mixing ratio as a piecewise function with `n_chunk` chunks.
   The chunk values are retrieved.
   """
   id : ClassVar[int] = 69
   apr_input_format : ClassVar[str] = \
   """
      n_pieces
      p_min
      p_max[0] vmr[0] vmr_err[0]
      p_max[1] vmr[1] vmr_err[1]
      ...
      p_max[n_pieces] vmr[n_pieces] vmr_err[n_pieces]
   
   """
   
   piecewise_profile : StateParam.using('Value of the piecewise profile', 'RATIO')
   
   n_pieces : ConstParam[int].using('Number of chunks in the piecewise profile')
   pressure_edges : ConstParam[np.ndarray].using('Pressure at the edges of each piecewise part', 'atm')
   
   
   

   @classmethod
   def from_apr(
      cls,
      f : IO,                                    # The open file descriptor of the *.apr file
      varident : np.ndarray[[3],int],            # The three "varident" integers that were last read from the *.apr file
      npro : int,                                # Number of altitude levels defined for the Atmosphere component
      ngas : int,                                # Number of gas volume mixing ratio profiles defined for the reference atmosphere
      ndust : int,                               # Number of aerosol species density profiles define for the reference atmosphere
      nlocations : int,                          # Number of locations defined for the atmosphere component
      runname : str,                             # Name of the *.apr file, without extension.
      sxminfac : float,                          # Minimum factor to bother calculating covariance matrix entries, below this value entries will be zero
      input_file_type : ArchNemesisFileTypeEnum, # Type of input files that we are reading the model from.
   ) -> Self:
      
      # Read from *.apr file
      n_peices = cls.read_apr_entries(f, (int,))
      p_edges = np.zeros((n_peices+1,))
      
      p_edges[0] = cls.read_apr_entries(f, (float,))
      p_edges[1:], xvals, xerrs = cls.read_apr_entries(f, (float, float, float), n_peices)
      
      # Construct model instance
      instance = cls.from_arrays(
         xvals,
         xerrs,
         n_peices,
         p_edges,
      )
      
      # Set model state parameter flags
      instance.piecewise_profile.log = True # `True` is the default, but set it here anyway
      instance.piecewise_profile.num_diff = True #  Always use numerical differentiation

      # return the model instance
      return instance

   def calculate_from_subprofretg(
      self,
      forward_model : "ForwardModel_0",
      ix : int,
      ipar : int,
      ivar : int,
      xmap : np.ndarray,
    ) -> None:
      atm = forward_model.AtmosphereX
      atm_profile_type, atm_profile_idx = atm.ipar_to_atm_profile_type(ipar)
      
      assert atm_profile_type == AtmosphericProfileTypeEnum.GAS_VOLUME_MIXING_RATIO
      
      self.pull_from_state_vector(forward_model.Variables.XN, forward_model.Variables.LX) # NOTE: this un-logs values if required
      
      atm, xmap1 = self.calculate(
         atm,
         atm_profile_type,
         atm_profile_idx,
      )
      
      forward_model.AtmosphereX = atm
      xmap[self.state_vector_slice, ipar, 0:atm.NP] = xmap1
   
   def calculate(
         self, 
         atm : "Atmosphere_0",
         #   Instance of Atmosphere_0 class we are operating upon
         
         atm_profile_type : AtmosphericProfileTypeEnum,
         #   ENUM of atmospheric profile type we are altering.
         
         atm_profile_idx : int | None,
         #   Index of the atmospheric profile we are altering (or None if the profile type does not have multiples)
   ) -> tuple["Atmosphere_0", np.ndarray]:
      p_bins = self.pressure_edges.v
      p_vals = self.piecewise_profile.v
   
      # update profile values
      temp = np.array(atm.VMR)
      fidx = np.digitize(atm.P, p_bins)
      
      temp[fidx] = p_vals[fidx]
      atm.edit_VMR(temp)
      
      # find the functional derivatives
      xmap = np.zeros((chunk.size,atm.VMR.size[0]),float)
      j=0
      for i in range(chunk.size):
         while fidx[j] < i+1:
            xmap[i, j] = 1
            j += 1

      # return results
      return atm, xmap
```